In [0]:
%sql
-- 1. All three layers are receiving data
SELECT 'bronze' AS layer, COUNT(*) AS rows, MAX(ingest_timestamp) AS latest FROM wiki_poc.poc.bronze_recentchange_raw
UNION ALL
SELECT 'silver', COUNT(*), MAX(ingest_timestamp) FROM wiki_poc.poc.silver_recentchange_enwiki
UNION ALL
SELECT 'gold',   COUNT(*), MAX(window_end)        FROM wiki_poc.poc.gold_page_edits_5min;


In [0]:
-- 2. Silver is ~5–15% of Bronze (rough filter-rate check)
SELECT
  ROUND(100.0 * 
    (SELECT COUNT(*) FROM wiki_poc.poc.silver_recentchange_enwiki) /
    (SELECT COUNT(*) FROM wiki_poc.poc.bronze_recentchange_raw), 1) AS silver_pct_of_bronze;


In [0]:
-- 3. Gold windows are closing (should see rows with window_end in the past)
SELECT window_start, window_end, COUNT(*) AS pages, SUM(edit_count) AS total_edits
FROM wiki_poc.poc.gold_page_edits_5min
GROUP BY window_start, window_end
ORDER BY window_start DESC LIMIT 5;

In [0]:
-- Data Consistency Check
WITH counts AS (
  SELECT
    (SELECT COUNT(*) FROM wiki_poc.poc.bronze_recentchange_raw)        AS bronze,
    (SELECT COUNT(*) FROM wiki_poc.poc.silver_recentchange_enwiki)     AS silver,
    (SELECT COUNT(*) FROM wiki_poc.poc.silver_recentchange_quarantine) AS quarantine,
    (SELECT SUM(edit_count) FROM wiki_poc.poc.gold_page_edits_5min)    AS gold_events
)
SELECT
  bronze,
  silver,
  quarantine,
  bronze - silver - quarantine                                    AS unaccounted,
  ROUND(100.0 * silver    / NULLIF(bronze, 0), 1)                AS silver_pct,
  ROUND(100.0 * quarantine / NULLIF(bronze, 0), 1)               AS quarantine_pct,
  gold_events,
  ROUND(100.0 * gold_events / NULLIF(silver, 0), 1)              AS gold_coverage_pct
FROM counts;

In [0]:
-- Dedup check
SELECT COUNT(*) AS duplicate_event_ids
FROM (
  SELECT event_id
  FROM wiki_poc.poc.silver_recentchange_enwiki
  GROUP BY event_id
  HAVING COUNT(*) > 1
);